<h1 align="center"><b>Gromacs Tutorials</b></h1>

<div align="center">
  <img src="https://github.com/SHEDOOMTC/Basic_Moco_2/raw/main/Assets/Moco_2_visuals.png" alt="Description" width="400"/>
</div>

<h2 style="text-align: left; color: darkred; font-weight: bold;">About and Usage</h2>

**[GROMACS](https://www.gromacs.org/about.html)** is a free, open-source software suite primarily used for performing molecular dynamics simulations. It is particularly well-suited for biochemical molecules like proteins, lipids, and nucleic acids, but its excellent performance in calculating non-bonded interactions has led many researchers to also use it for non-biological systems such as polymers. The project originated in 1991 at the University of Groningen in the Netherlands, originally named "GROningen MAchine for Chemical Simulations," and has since been maintained by developers from universities and research institutions worldwide. 

In this tutorial, we will look at how to perform MD Simulations using gromacs.

This is a practical session follow up of the course on **"[Basic Molecular Modelling Course for Chemists and Life Sciences II](https://github.com/SHEDOOMTC/Basic_Moco_2)** ***(Basic Moco II)***"

Cite this tutorial as **"""""**

In [ ]:
#Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nglview as nv
import mdtraj as md

<h3 style="text-align: left; color: darkblue; font-weight: bold;">The GMX Global command</h3>

Gromacs is organized around the **`GMX`** global command. Since GROMACS 5.0, it serves as the single entry point for invoking virtually all GROMACS tools, replacing the older system of standalone executables. The gmx command acts as a wrapper or launcher. You use it by typing gmx followed by a specific subcommand (the tool you want to run) and then any options for that tool.

In [ ]:
#Lets explore the gmx command
!gmx -h

<h2 style="text-align: left; color: darkred; font-weight: bold;">System Preparations</h2>

Like in every other MD simulation tool, we start off by preparing our proteins using a chosen forcefield

In [ ]:
# Download our protein of interest from the RCSB PDB database
!wget https://files.rcsb.org/download/3HTB.pdb

# Lets clean the pdb file and remove water molecules and heteroatoms
!grep "^ATOM" 3HTB.pdb > 3HTB_protein.pdb
!grep "^HETATM" 3HTB.pdb | grep "JZ4 A" > JZ4.pdb

**Visualize them**

In [ ]:
!cat JZ4.pdb

<h3 style="text-align: left; color: darkblue; font-weight: bold;">gmx pdb2gmx</h3>

The **`pdb2gmx`** command is interactive and will ask you to select a force field. For this practical, please select the following options:

`AMBER99SB-ILDN` force field (option 6)

For simplicity, it can be redirected using “echo” command

In [ ]:
#Using the echo command to select the force field and water model automatically
!echo 6/n | gmx pdb2gmx -f 3HTB_protein.pdb -o 3HTB.gro -water spce

This step produces the A **`posre.itp`** (position restraint) and **`topol.top`** (topology) is generated in addition to **`gro file`**

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Ligand Preparations-- ACYPYPE</h3>

The Force fields below are available for use to parametrize small molecules. We will use the AMBER forcefield through **[Antechamber]()** wrapped up in **[Acpype]()**. Acpype has been pre-installed together with gromacs

| Force Field | Tool / Server | Description |
| :--- | :--- | :--- |
| **AMBER** | **Antechamber** | Parametrizes molecules using GAFF |
| **AMBER** | **acpype** | A Python interface to Antechamber, writes GROMACS topologies |
| **CHARMM** | **CGenFF** | The official CHARMM General Force Field server |
| **GROMOS96** | **ATB** | A newer server for topology generation, uses GROMOS96 54A7 |
| **OPLS-AA** | **LigParGen** | A server from the Jorgensen group to produce OPLS topologies |

**See all the possible input for acpype**

In [ ]:
!acpype -h

In [ ]:
#The -o gmx controls which parameters to output
!acpype -i JZ4.pdb -b JZ4 -n 0 -c bcc -a gaff2 -o gmx

**Inspect the output file and copy out the gromacs files needed for the simulations**

We are interested in the **`JZ4_GMX.gro, JZ4_GMX.itp, JZ4_GMX.top`** and **`posre_JZ4.itp`** files. In many cases **`JZ4_GMX.itp`**, the contains the atom topologies while **`JZ4_GMX.top`** contain only the groups and pointers to the **`itp`**. This will have implications when combining the topologies.

In [ ]:
%cd JZ4.acpype
!ls

In [ ]:
#Copy out all the files with "GMX" names and the posre_JZ4.itp
!cp *GMX* posre_JZ4.itp ../
%cd ../

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Creating a Complex Topology</h3>


Creating a complex topology involves combining both the ligand and protein topology into a single **`TOP file (top.top)`** and calling their respective **`ITP`** files within the single file. *Usually, this is done manually, but I had written a script to automate that.*

In [ ]:
#Get the Script
!git clone https://github.com/SHEDOOMTC/Combine_Topo.git

#Copy out the script to the current directory
!cp Combine_Topo/combine_topo.sh .

#Usage of the script
!bash combine_topo.sh

In [ ]:
#Inspect the JZ4_GMX.top file
!cat JZ4_GMX.top

So, we will use the **`JZ4_GMX.itp`** for merging the **`topology`**

In [ ]:
!bash combine_topo.sh topol.top JZ4_GMX.itp 3HTB.itp JZ4.itp posre_JZ4.itp JZ4

Now, we have a single **`top.top`** file with all the other topology files (**`JZ4_GMX.itp, atomtypes.itp, 3HTB.itp`**) referenced inside it

In [ ]:
!head -n  20 top.top

<h3 style="text-align: left; color: darkblue; font-weight: bold;">Creating a Complex GRO file</h3>

Creating a complex GRO file involves combining both the ligand and protein gro into a single **`GRO file (complex.gro)`**. *Usually, this is done manually, but I had written a script to automate that*. It was initially written for DNA and ligand but it works for proteins as well

In [ ]:
#Get the Script
!git clone https://github.com/SHEDOOMTC/okammuta-na-gromacs.git

#Copy out the script to the current directory
!cp okammuta-na-gromacs/njiko-DNA-na-ligand.sh .
!chmod +x njiko-DNA-na-ligand.sh

#Usage of the script
!bash njiko-DNA-na-ligand.sh

In [ ]:
#Copy the 3HTB.gro to a new file called complex.gro
!cp 3HTB.gro complex.gro

#Usage of the script. We will pass 1 to the script to indicate how many ligands we want to add. In this case we have only one ligand JZ4
!echo 1 | bash njiko-DNA-na-ligand.sh JZ4_GMX.gro complex.gro

In [ ]:
#Inspect the difeerence between 3HTB.gro and complex.gro
!diff -y 3HTB.gro complex.gro

The `total atom count` at the beginning of a file has increased; also scrolling down shows that the `ligand` has been appended

<h2 style="text-align: left; color: darkred; font-weight: bold;">Index and Position Restraint Files</h2>

<h3 style="text-align: left; color: darkblue; font-weight: bold;">gmx make_ndx</h3>

`Index file` contain a group of `atom indices`. Useful for selection and applying `unique effects` to a group. We can generate a a file with only the atoms of our `ligand JZ4` to allow us apply any effect we want on the ligand alone. `gmx make_ndx` opens an `interactive interface` for you to choose or create specific groups. Gromacs have `custom groups` pre-defined; but you can create yours as a combination of any atom, residues or groups of residues.

- **We will create a `global index file` that includes all the atoms in the complex**

In [ ]:
#Generate the index file and visualize the first 8 lines of the index file
!echo -e "0\nq" | gmx make_ndx -f complex.gro -o index.ndx

!head -n 8 index.ndx

<h3 style="text-align: left; color: darkblue; font-weight: bold;">gmx genrestr</h3>

**`Position restraint`** files encodes `restraint forces` to be applied to specified atoms or groups of atoms. So, it requires to be generated using `index files` for the group of interest.

Apply it by calling with **“`ifdef POSRES`”** in the `topology file`, then call with **“`define = -DPOSRES`”** in the **`MDP`** file

During System preparations, `restraint files` for the `ligand` and `protein` have been generated. Lets check them both in our folder, and topology files

In [ ]:
#Generate a position restraint for the whole system with force contant of 10
#!echo -e "0\n" | gmx genrestr -f protein.gro -n index.ndx -o posre_custom.itp -fc 10

In [ ]:
!ls -l posre*

In [ ]:
!grep -rn -C 3 "POSRES" top.top

So, we have **`#ifdef POSRES_LIG`** for the **`ligand`** and **`#ifdef POSRES`** for the **`protein`** and optionally **`#ifdef POSRES_WATER`** for the water. Take note as we will define these in the **`MDP`** files during the steps below

<h2 style="text-align: left; color: darkred; font-weight: bold;">Solvation</h2>

<h3 style="text-align: left; color: darkblue; font-weight: bold;">gmx editconf</h3>

The complex must be put into a simulation box to allow us enforce period boundaries. **`gmx editconf`** is a versatile GROMACS utility used to manipulate molecular structure files before simulation. It can convert between coordinate formats, define or modify simulation boxes, centre molecules, rotate structures, and translate coordinates. Because it rewrites the structure file, it is also commonly used to clean up or renumber residue information in .gro files.

**`-center`** option can be used to shift the geometric center of the system.

In [ ]:
#To learn more about the gmx editconf command, you can use the following command:
!gmx editconf -h

In [ ]:
#We will insert our system into a cubic box of size 1.2 nm from the solute to the box edge. The -c flag will center the solute in the box.
!gmx editconf -f complex.gro -o complex_box.gro -c -d 1.2 -bt cubic

In [ ]:
#Visualize the box
traj = md.load("complex_box.gro")  # Use your actual .gro filename
view = nv.show_mdtraj(traj)
view.clear_representations()
view.add_representation("ball+stick")  # Note: 3-residue peptide is too short for "cartoon"
view.add_unitcell()
view

<h3 style="text-align: left; color: darkblue; font-weight: bold;">gmx solvate</h3>

We will fill the simulation box with solvent molecules (most commonly water) around a solute. During execution, **`gmx solvate`** tiles a pre-equilibrated solvent box—such as **`spc216.gro`** for **`3-point water`** models—across the dimensions defined at the bottom of your input coordinate file, then automatically deletes any solvent molecules that overlap with your solute atoms based on van der Waals radii. 

In doing this, we must pass the **`top.top`** file so it gets updated with the number of wate added to the box. The **`-maxsol`** flag can control how many solvents you want to add (perhaps to match another simulation)

In [ ]:
!gmx solvate -cp complex_box.gro -cs spc216.gro -o complex_solvated.gro -p top.top

#Visualize the water box
traj = md.load("complex_solvated.gro")  # Use your actual .gro filename
view = nv.show_mdtraj(traj)
view.clear_representations()
view.add_representation("ball+stick")  # Note: 3-residue peptide is too short for "cartoon"
view.add_unitcell()
view

<h2 style="text-align: left; color: darkred; font-weight: bold;">Ionization</h2>

<h3 style="text-align: left; color: darkblue; font-weight: bold;">gmx grompp</h3>

**`gmx grompp`** is a command-line tool that compiles your simulation inputs into a single portable binary run file **`(.tpr)`**. It reads your atomic coordinates **`(.gro)`**, system topology and force field parameters **`(.top)`**, and simulation settings **`(.mdp)`** *(see shortly)*, checks that atom counts, names, and physics parameters match across all files, and resolves preprocessor directives like position restraints. The resulting **`.tpr`** file contains everything **`gmx mdrun`** *(see shortly)* needs to execute energy minimization, equilibration, or a production molecular dynamics simulation.

The **`MDP`** file is used for ionization but the contents are not necessary. It will become important from minimization and further steps

<h3 style="text-align: left; color: darkblue; font-weight: bold;">gmx genion</h3>

**`gmx genion`** replaces randomly selected solvent molecules in your simulation box with monoatomic ions **`(such as Na⁺ or Cl⁻)`**. It reads your compiled **`.tpr`** file to determine the net charge of your system, swaps out water molecules **`(SOL)`** for the requested ions to either **`neutralize`** the system or reach a **`target salt concentration`**, and updates your topology file **`(top.top)`** with the new solvent and ion counts.

- **Compile a `minimalist MDP file`**

In [ ]:
%%bash
cat > ions.mdp << EOF
integrator  = steep
nsteps      = 0
cutoff-scheme = Verlet
EOF

We will add **`NA and CL`** to reach a concentration of **`0.15`**. Since Ions replace **`SOL`**, we need to pass it via **`echo or printf`**

In [ ]:
!gmx grompp -f ions.mdp -c complex_solvated.gro -p top.top -o ions.tpr

!printf "SOL\n" | gmx genion -s ions.tpr -o complex_ions.gro -p top.top -pname NA -nname CL -neutral -conc 0.15


In [ ]:
#Visualize the water box
traj = md.load("complex_ions.gro")  # Use your actual .gro filename
view = nv.show_mdtraj(traj)
view.clear_representations()
view.add_representation("cartoon", selection="protein")
view.add_representation("licorice", selection="JZ4")  # Your ligand
view.add_representation("spacefill", selection="NA or CL")
view.add_representation("line", selection="water", opacity=0.5)
view.add_unitcell()
view

<h2 style="text-align: left; color: darkred; font-weight: bold;">Energy Minimization</h2>

Before we can begin dynamics, we must ensure that the system has no steric clashes or inappropriate geometry. As usual, we compile the **`TPR`** file using the grommp command and a correct MDP file

Lets discuss the **`MDP file`** first and the **`gmx mdrun`** command


<h3 style="text-align: left; color: darkblue; font-weight: bold;">The MDP File</h3>

An **`.mdp` (Molecular Dynamics Parameters)** file is a plain-text configuration file used by GROMACS to define **how a simulation runs**. While your coordinate file (`.gro`) specifies *where* the atoms are and your topology file (`.top`) specifies *what* the atoms and bonds are, the `.mdp` file controls the **physics, algorithms, and runtime settings** of the simulation. Before running a simulation, `gmx grompp` combines the `.mdp`, `.gro`, and `.top` files into a single binary run input file (`.tpr`).

<h3 style="text-align: left; color: darkblue; font-weight: bold;">gmx mdrun</h3>

**`gmx mdrun`** is the main computational engine of GROMACS that executes **`molecular dynamics simulations`**, **`energy minimization`**, and **`free-energy calculations`**. It reads the portable binary run input file (`.tpr`) generated by `gmx grompp` and integrates Newton's equations of motion over time across available CPU cores and GPUs.

In [ ]:
# Check the inputs of gmx mdrun
!gmx mdrun -h

- We will run energy minimization for **`10000`** steeps using the **`steep algorithm`**. We will activate **`restraints`** on the protein and the ligand by including **`-DPOSRES and -DPOSRES_LIG`** in the **`MDP`** file. Positional restraint is only applied if the **`-r flag`** is passed during the **`grompp`**

In [ ]:
%%bash
cat > minim.mdp << EOF
; 1. Run Control
define      = -DPOSRES -DPOSRES_LIG
integrator  = steep         ; Steepest descent energy minimization algorithm
emtol       = 1000.0        ; Stop minimization when max force < 1000.0 kJ/mol/nm
emstep      = 0.01          ; Initial step size (nm)
nsteps      = 10000         ; Maximum number of minimization steps

; 2. Neighbor Searching & Electrostatics
nstlist         = 10        ; Frequency to update the neighbor list
cutoff-scheme   = Verlet    ; Buffered neighbor searching scheme
coulombtype     = PME       ; Particle Mesh Ewald for long-range electrostatics
rcoulomb        = 1.0       ; Short-range electrostatic cutoff (nm)
rvdw            = 1.0       ; Short-range van der Waals cutoff (nm)

; 3. Periodic Boundary Conditions
pbc             = xyz       ; Periodic Boundary Conditions in all 3 dimensions
EOF

In [ ]:
!gmx grompp -f minim.mdp -c complex_ions.gro -r complex_ions.gro -p top.top -o em.tpr

In [ ]:
!gmx mdrun -v -deffnm em

Many files were produced after the run including **`em.edr, em.log, em.gro`**

In [ ]:
!ls em*

- **Was our minimization successful?**

We will run **`gmx energy`** on the **`em.edr`** file to extract the energy and make a plot of it. The **`EDR`** file contain many state variables from the run with numberings, so we will select a number (or the variable) to write out the energy files 


In [ ]:
!echo -e "Potential\n0" | gmx energy -f em.edr -o potential.xvg

In [ ]:
#Make the plot of the potential energy
data = np.loadtxt("potential.xvg", comments=["#", "@"])
steps = data[:, 0]
potential = data[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(steps, potential, color="#1f77b4", linewidth=1.5)

plt.title("Energy Minimization: Potential Energy", fontsize=14, fontweight="bold")
plt.xlabel("Energy Minimization Step", fontsize=12)
plt.ylabel("Potential Energy (kJ/mol)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("potential_energy.png", dpi=300)
plt.show()

<h2 style="text-align: left; color: darkred; font-weight: bold;">NVT Equilibration</h2>

**NVT equilibration (canonical ensemble)** is a short molecular dynamics simulation phase that brings your system to a **`target temperature`** while keeping the number of **particles (N)** and **simulation box volume (V)** constant. Following energy minimization, **`gmx mdrun`** assigns initial **random velocities** to the atoms and uses a **thermostat (such as V-rescale or Nose-Hoover)** to stabilize the temperature—typically at 300 K—while **position restraints** hold the heavy atoms of your solute in place so the surrounding water and ions can relax naturally around it without distorting its structure.

- **Generate the `MDP file`**

We will heat our system from **`0 to 300K`** for **`10000 steps`** using a **`timestep of 2fs`** while still maintaining the **`restraints`**

In [ ]:
%%bash
cat > nvt.mdp << EOF
define                  = -DPOSRES -DPOSRES_LIG
integrator              = md
nsteps                  = 10000     ; 20 ps
dt                      = 0.002     ; 2 fs
nstenergy               = 500
nstlog                  = 500
nstxout-compressed      = 500
constraints             = h-bonds
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
tcoupl                  = V-rescale
tc-grps                 = System
tau_t                   = 0.1    
ref_t                   = 300    
gen_vel                 = yes
gen_temp                = 300
EOF

In [ ]:
#Run the grompp command and the nvt equilibration.
!gmx grompp -f nvt.mdp -c em.gro -r em.gro -p top.top -o nvt.tpr
!gmx mdrun -deffnm nvt

- **Was our heating successful?**

We will run **`gmx energy`** on the **`nvt.edr`** file to extract the temperature and make a plot of it. The **`EDR`** file contain many state variables from the run with numberings, so we will select a number (or the variable) to write out the temperature files 

In [ ]:
!echo -e "Temperature\n0" | gmx energy -f nvt.edr -o Temperature.xvg

In [ ]:
#Make the plot of the Temperature
data = np.loadtxt("Temperature.xvg", comments=["#", "@"])
steps = data[:, 0]
potential = data[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(steps, potential, color="#1f77b4", linewidth=1.5)

plt.title("NVT Equilibration: Temperature", fontsize=14, fontweight="bold")
plt.xlabel("Time (ps)", fontsize=12)
plt.ylabel("Temperature (K)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("Temperaturey.png", dpi=300)
plt.show()

<h2 style="text-align: left; color: darkred; font-weight: bold;">NPT Equilibration</h2>

Here, we will stabilize the `system's pressure` and `density` while keeping the `particle count (N)`, `pressure (P)`, and `temperature (T)` constant. Using both a `thermostat` and a `barostat`, GROMACS dynamically scales the simulation box dimensions until the internal pressure matches your target—typically 1 bar—and the water density settles around ~1000 kg/m³. Position restraints remain active on the solute's heavy atoms so its structure stays intact while the solvent box adjusts around it before you release restraints for production MD.

- **Generate the `MDP file`**

We will equilibrate our system to **`1 atm`** for **`20000 steps`** using a **`timestep of 2fs`** while still maintaining the **`restraints`**. Since this is a continuation, we set **`gen_vel = no`** or **`continuation = yes`**

In [ ]:
%%bash
cat > npt.mdp << EOF
define                  = -DPOSRES -DPOSRES_LIG
continuation            = yes
integrator              = md
nsteps                  = 20000     ; 100 ps
dt                      = 0.002     ; 2 fs
nstenergy               = 500
nstlog                  = 500
nstxout-compressed      = 500
constraints             = h-bonds
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
tcoupl                  = V-rescale
tc-grps                 = System
tau_t                   = 0.1    
ref_t                   = 300     
pcoupl                  = C-rescale
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
refcoord_scaling        = com
gen_vel                 = no
EOF


In [ ]:
#Run the grompp command and the npt equilibration.
!gmx grompp -f npt.mdp -c nvt.gro -r nvt.gro -p top.top -o npt.tpr
!gmx mdrun -deffnm npt

- **Was our equilibration successful?**

We will run **`gmx energy`** on the **`npt.edr`** file to extract the pressure and make a plot of it. The **`EDR`** file contain many state variables from the run with numberings, so we will select a number (or the variable) to write out the pressure files 

In [ ]:
!echo -e "Pressure\n0" | gmx energy -f npt.edr -o Pressure.xvg

In [ ]:
#Make the plot of the Pressure
data = np.loadtxt("Pressure.xvg", comments=["#", "@"])
steps = data[:, 0]
potential = data[:, 1]

plt.figure(figsize=(8, 5))
plt.plot(steps, potential, color="#1f77b4", linewidth=1.5)

plt.title("NPT Equilibration: Pressure", fontsize=14, fontweight="bold")
plt.xlabel("Time (ps)", fontsize=12)
plt.ylabel("Pressure (bar)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("Pressure.png", dpi=300)
plt.show()

<h2 style="text-align: left; color: darkred; font-weight: bold;">Production Run</h2>

The `production run` is the final, `unrestrained` molecular dynamics simulation phase where you collect actual trajectory data to study your system's behavior. After `NVT` and `NPT` equilibration have stabilized the `temperature`, `pressure`, and `density`, you remove the `position restraints (-DPOSRES)` so your solute can move and sample conformations freely under realistic physiological conditions. Running over longer timescales (`nanoseconds to microseconds`), this stage generates the compressed trajectory (`.xtc`) and energy `(.edr`) files used for post-simulation analyses like `RMSD, Ramachandran plots, hydrogen bonding, and free energy landscapes`.

We will run our system to for **`2000000 steps`** using a **`timestep of 2fs`**. The **`restraints`** were removed. Since this is a continuation, we set **`gen_vel = no`** or **`continuation = yes`**

In [ ]:
%%bash
cat > md.mdp << EOF
integrator              = md
nsteps                  = 2000000   ; 4 ns (2 fs * 2,000,000)
dt                      = 0.002     ; 2 fs
nstenergy               = 5000      
nstlog                  = 5000       
nstxout-compressed      = 5000      
constraints             = h-bonds
coulombtype             = PME
rcoulomb                = 1.0
rvdw                    = 1.0
tcoupl                  = V-rescale
tc-grps                 = System
tau_t                   = 0.1     
ref_t                   = 300     
pcoupl                  = Parrinello-Rahman
pcoupltype              = isotropic
tau_p                   = 2.0
ref_p                   = 1.0
compressibility         = 4.5e-5
continuation            = yes
gen_vel                 = no
EOF

In [ ]:
#Run the grompp and the production run
!gmx grompp -f md.mdp -c npt.gro -r npt.gro -p top.top -o md.tpr
!gmx mdrun -deffnm md

<h2 style="text-align: left; color: darkred; font-weight: bold;">Controlling Hardware Resources (CPUs & GPUs)</h2>

For larger production runs, you can explicitly assign thread counts and GPU acceleration:

```bash
gmx mdrun -deffnm md -ntmpi 1 -ntomp 8 -nb gpu
```
* **`-ntmpi 1`:** Uses 1 thread-MPI rank.
* **`-ntomp 8`:** Assigns 8 OpenMP threads to that rank.
* **`-nb gpu`:** Offloads short-range non-bonded calculations to an available GPU.

<h2 style="text-align: left; color: darkred; font-weight: bold;">Resuming an Interrupted Simulation</h2>

If a simulation stops early due to a power outage or cluster wall-time limit, resume it cleanly using the checkpoint file (`.cpt`):

```bash
gmx mdrun -deffnm md -cpi md.cpt -append
```
* **`-cpi md.cpt`:** Loads the exact state from the last saved checkpoint.
* **`-append`:** Continues writing directly to your existing `.log`, `.edr`, and `.xtc` files.

<h2 style="text-align: left; color: darkred; font-weight: bold;">Reflections</h2>

Through this `molecular dynamics workflow`, you have learned how to take a `protein` and `ligand` to a fully equilibrated, unrestrained production simulation in GROMACS. 

Setting up the system highlighted how each preparation step builds directly on the previous one, starting with `parametrizing` the protein and ligands, building the `complex GRO file` and `topologies` before solvating the solute (`gmx solvate`) and adding neutralizing ions (`gmx genion`). 

Compiling inputs with `gmx grompp` prior to executing `gmx mdrun`. Running the simulation in stages demonstrated why gradual equilibration is essential: `energy minimization` resolved steric clashes as confirmed by a plateau in potential energy, `NVT equilibration` stabilized the temperature at `300 K` while position restraints held the peptide steady, and `NPT equilibration` allowed the `pressure` and `box density` to settle at 1 bar before releasing restraints for the production run. 

<h2 style="text-align: left; color: darkred; font-weight: bold;">References</h2>

1. http://www.mdtutorials.com/gmx/
2. http://www.gromacs.org/
